<a href="https://colab.research.google.com/github/Vijay-1010110/FaceKeyAnimation/blob/main/notebooks/1_Colab_Data_Collector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 FaceKey Animation Studio — Google Colab Cloud Data Collector
### Automated Silent YouTube Extraction + Google Drive (5 TB Plan) Chunking

**Features:**
- **5 TB Storage Optimized**: Seamlessly uploads `.tar.gz` chunks into your 5 TB Google Drive.
- **Parallel Multi-Worker Coordination**: Can run concurrently with Kaggle and Local PC workers without duplicate downloads!
- **Distributed Atomic Locks**: Locks URLs in Google Drive (`locks/<id>.lock.json`) with live heartbeats.
- **Auto-Reclaims Stale Locks**: If a session terminates unexpectedly or disconnects, abandoned locks are automatically recovered and resumed!
- **Zero Local Disk Bloat**: Local scratch memory is purged after each chunk to stay strictly under 5 GB.

## Step 1: Mount Google Drive (5 TB Plan)
Run this cell and click **Connect to Google Drive**.

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/FaceKeyDataset'
os.makedirs(f'{DRIVE_DIR}/chunks', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/locks', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)
print(f'[+] Google Drive mounted successfully at: {DRIVE_DIR}')

Mounted at /content/drive
[+] Google Drive mounted successfully at: /content/drive/MyDrive/FaceKeyDataset


## Step 2: Clone FaceKey Repository & Install Fast Cloud Dependencies

In [14]:
%cd /content
!rm -rf /content/FaceKeyAnimation
!git clone https://github.com/Vijay-1010110/FaceKeyAnimation.git /content/FaceKeyAnimation
%cd /content/FaceKeyAnimation
!pip install -q yt-dlp mediapipe opencv-python-headless numpy sounddevice mss av

/content
Cloning into '/content/FaceKeyAnimation'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 164 (delta 58), reused 141 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 3.44 MiB | 31.98 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/FaceKeyAnimation


In [15]:
# Clone or pull latest code from your GitHub repository
REPO_URL = 'https://github.com/Vijay-1010110/FaceKeyAnimation.git'
APP_DIR = '/content/FaceKeyAnimation'

if os.path.exists(APP_DIR):
    !cd {APP_DIR} && git pull
else:
    !git clone {REPO_URL} {APP_DIR}

%cd {APP_DIR}

# Install lightweight headless runtime dependencies
!pip install -q yt-dlp mediapipe opencv-python-headless numpy sounddevice

Already up to date.
/content/FaceKeyAnimation


## Step 3: Inspect / Add YouTube URLs in Your Colab Queue
Colab uses its dedicated queue `youtube_urls_colab.txt` inside `MyDrive/FaceKeyDataset/`.
You can paste links here or directly edit the text file in Google Drive. Even while running, new links are detected dynamically!

In [3]:
urls_file = f'{DRIVE_DIR}/youtube_urls_colab.txt'

if not os.path.exists(urls_file):
    # Copy from repo if not on drive yet
    import shutil
    repo_colab = os.path.join(APP_DIR, 'sessions', 'youtube_urls_colab.txt')
    if os.path.exists(repo_colab):
        shutil.copy2(repo_colab, urls_file)

print(f'[+] Colab Queue file ready on Google Drive: {urls_file}')
with open(urls_file, 'r', encoding='utf-8') as f:
    lines = [l.strip() for l in f if l.strip() and not l.startswith('#')]
    print(f'[*] Active URLs in Colab Queue: {len(lines)} videos queued')
    print('First 5 links:')
    for u in lines[:5]:
        print('  -', u)

[+] Colab Queue file ready on Google Drive: /content/drive/MyDrive/FaceKeyDataset/youtube_urls_colab.txt
[*] Active URLs in Colab Queue: 170 videos queued
First 5 links:
  - https://www.youtube.com/watch?v=APCWOBUZNjc
  - https://www.youtube.com/watch?v=whPIzfm-Xng
  - https://www.youtube.com/watch?v=cVLpdzhcU0g
  - https://www.youtube.com/watch?v=mq10ZvZgxy4
  - https://www.youtube.com/watch?v=9fpztV4oMd4


## Step 4: Run Silent 480p Cloud Batch Data Collector
This runs completely in the background on Google Colab:
- Identified as `colab-worker-1`.
- Claims locks in Drive before processing to prevent overlap with Kaggle.
- Heartbeat keeps lock active; auto-reclaims stale locks if interrupted.
- Packages data into compressed `.tar.gz` chunks in your 5 TB Google Drive.
- Cleans local Colab disk to maintain < 5 GB usage.

In [ ]:
!python scripts/cloud_data_collector.py \
    --drive-dir "/content/drive/MyDrive/FaceKeyDataset" \
    --worker-id "colab-worker-1" \
    --urls-file "/content/drive/MyDrive/FaceKeyDataset/youtube_urls_colab.txt" \
    --chunk-size 5 \
    --quality 480p

 [CLOUD] FACEKEY MULTI-WORKER CLOUD COLLECTOR & DISTRIBUTED LOCK MANAGER
[+] Google Drive successfully mounted at: /content/drive/MyDrive/FaceKeyDataset
[*] Worker ID         : colab-worker-1
[*] Google Drive Root : /content/drive/MyDrive/FaceKeyDataset
[*] Chunks Storage    : /content/drive/MyDrive/FaceKeyDataset/chunks
[*] Active Locks Dir  : /content/drive/MyDrive/FaceKeyDataset/locks
[*] Shared DB Path    : /content/drive/MyDrive/FaceKeyDataset/data/shared_completed_sources.json
[!] Cleaned stale lock: Q8QxTRI72js.lock.json (idle 21.0m)
[+] Auto-reclaimed 1 stale lock(s) from terminated/expired sessions!
[*] Active URLs Queue : /content/drive/MyDrive/FaceKeyDataset/youtube_urls_colab.txt
------------------------------------------------------------------------------------
[*] Queue File Status  : 170 Total | 0 Done | 170 Pending
[*] Global Multi-Cloud : 0 Videos Completed (0.00 hrs across all workers)
[*] Google Drive Plan  : 5 TB Capacity (Storage headroom is virtually unlimited)

